# 00_creacion_objetos

Crea los objetos lógicos del pipeline Smart Data.

Este notebook **no crea infraestructura base** como catálogo, schema, storage credentials, external locations o volumes.  
Esos objetos ya deben existir como parte del setup inicial.

Responsabilidad de este notebook:

- Recibir `catalog_name`, `schema_name` y `storage_account`.
- Usar solo los volumes `raw` y `backup`.
- Crear carpetas operativas mínimas para Auto Loader y backup.
- Crear tablas Delta external/unmanaged usando `LOCATION`.
- Crear vistas de consumo para dashboard.

Convención de rutas:

- Archivos de entrada: `/Volumes/{catalog_name}/{schema_name}/raw`
- Archivos procesados: `/Volumes/{catalog_name}/{schema_name}/backup`
- Tablas bronze: `abfss://bronze@{storage_account}.dfs.core.windows.net/tables/...`
- Tablas silver: `abfss://silver@{storage_account}.dfs.core.windows.net/tables/...`
- Tablas gold: `abfss://gold@{storage_account}.dfs.core.windows.net/tables/...`
- Metadata Auto Loader: `abfss://bronze@{storage_account}.dfs.core.windows.net/metadata/smart_data_ingestion/...`


In [0]:
# COMMAND ----------

dbutils.widgets.text("catalog_name", "smart_data_dev", "Catalog name")
dbutils.widgets.text("schema_name", "storage", "Schema name")
dbutils.widgets.text("storage_account", "storagetoxdev", "Storage account")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
storage_account = dbutils.widgets.get("storage_account")

print(f"catalog_name: {catalog_name}")
print(f"schema_name: {schema_name}")
print(f"storage_account: {storage_account}")


In [0]:
# COMMAND ----------

# Volumes: only raw and backup are used.
raw_base_path = f"/Volumes/{catalog_name}/{schema_name}/raw"
backup_base_path = f"/Volumes/{catalog_name}/{schema_name}/backup"

# External table locations.
# These paths must NOT overlap with volume paths.
bronze_table_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/tables"
silver_table_base_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/tables"
gold_table_base_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/tables"

# Auto Loader technical metadata.
# These paths are separated from both raw files and Delta table locations.
checkpoint_base_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
    "metadata/smart_data_ingestion/checkpoints"
)

schema_base_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
    "metadata/smart_data_ingestion/schemas"
)

print("Volume paths:")
print(f"raw_base_path: {raw_base_path}")
print(f"backup_base_path: {backup_base_path}")

print("External table paths:")
print(f"bronze_table_base_path: {bronze_table_base_path}")
print(f"silver_table_base_path: {silver_table_base_path}")
print(f"gold_table_base_path: {gold_table_base_path}")

print("Auto Loader metadata paths:")
print(f"checkpoint_base_path: {checkpoint_base_path}")
print(f"schema_base_path: {schema_base_path}")


In [0]:
# COMMAND ----------

# Minimal operational folders.
# Delta table folders are created by Delta when the tables are created or written.
for path in [
    f"{checkpoint_base_path}/raw_to_bronze",
    f"{schema_base_path}/raw_to_bronze",
    f"{backup_base_path}/processed"
]:
    dbutils.fs.mkdirs(path)

print("Carpetas operativas creadas o ya existentes.")


In [0]:
# COMMAND ----------

control_archivos_path = f"{bronze_table_base_path}/control_archivos"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{schema_name}`.`control_archivos` (
    nombre_archivo STRING,
    archivo_origen STRING,
    registros_bronze BIGINT,
    registros_invalidos_bronze BIGINT,
    registros_silver BIGINT,
    registros_error BIGINT,
    fecha_ejecucion TIMESTAMP,
    id_proceso STRING,
    detalle STRING
)
USING DELTA
LOCATION '{control_archivos_path}'
""")

print(f"Tabla creada o validada: {catalog_name}.{schema_name}.control_archivos")
print(f"LOCATION: {control_archivos_path}")


In [0]:
# COMMAND ----------

bronze_reportes_path = f"{bronze_table_base_path}/bronze_reportes"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{schema_name}`.`bronze_reportes` (
    email STRING,
    jyv STRING,
    badmail STRING,
    baja STRING,
    fecha_envio STRING,
    fecha_open STRING,
    opens STRING,
    opens_virales STRING,
    fecha_click STRING,
    clicks STRING,
    clicks_virales STRING,
    links STRING,
    ips STRING,
    navegadores STRING,
    plataformas STRING,
    _corrupt_record STRING,
    _rescued_data STRING,
    archivo_origen STRING,
    nombre_archivo STRING,
    id_proceso STRING,
    fecha_ejecucion TIMESTAMP
)
USING DELTA
LOCATION '{bronze_reportes_path}'
""")

print(f"Tabla creada o validada: {catalog_name}.{schema_name}.bronze_reportes")
print(f"LOCATION: {bronze_reportes_path}")


In [0]:
# COMMAND ----------

silver_reportes_path = f"{silver_table_base_path}/silver_reportes"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{schema_name}`.`silver_reportes` (
    email STRING,
    jyv STRING,
    badmail STRING,
    baja STRING,
    fecha_envio TIMESTAMP,
    fecha_open TIMESTAMP,
    opens STRING,
    opens_virales STRING,
    fecha_click TIMESTAMP,
    clicks STRING,
    clicks_virales STRING,
    links STRING,
    ips STRING,
    navegadores STRING,
    plataformas STRING,
    archivo_origen STRING,
    nombre_archivo STRING,
    id_proceso STRING,
    fecha_ejecucion TIMESTAMP,
    fecha_envio_mes STRING
)
USING DELTA
PARTITIONED BY (fecha_envio_mes)
LOCATION '{silver_reportes_path}'
""")

print(f"Tabla creada o validada: {catalog_name}.{schema_name}.silver_reportes")
print(f"LOCATION: {silver_reportes_path}")


In [0]:
# COMMAND ----------

errores_path = f"{silver_table_base_path}/errores"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{schema_name}`.`errores` (
    email STRING,
    jyv STRING,
    badmail STRING,
    baja STRING,
    fecha_envio STRING,
    fecha_open STRING,
    opens STRING,
    opens_virales STRING,
    fecha_click STRING,
    clicks STRING,
    clicks_virales STRING,
    links STRING,
    ips STRING,
    navegadores STRING,
    plataformas STRING,
    archivo_origen STRING,
    nombre_archivo STRING,
    id_proceso STRING,
    registro_original STRING,
    rescued_data_column STRING,
    razon_error STRING,
    fecha_ejecucion TIMESTAMP
)
USING DELTA
LOCATION '{errores_path}'
""")

print(f"Tabla creada o validada: {catalog_name}.{schema_name}.errores")
print(f"LOCATION: {errores_path}")


In [0]:
# COMMAND ----------

maestro_visitante_path = f"{gold_table_base_path}/maestro_visitante"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{schema_name}`.`maestro_visitante` (
    email STRING,
    fecha_primera_visita DATE,
    periodo_baja STRING,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
LOCATION '{maestro_visitante_path}'
""")

print(f"Tabla creada o validada: {catalog_name}.{schema_name}.maestro_visitante")
print(f"LOCATION: {maestro_visitante_path}")


In [0]:
# COMMAND ----------

visitas_mensuales_path = f"{gold_table_base_path}/visitas_mensuales"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{schema_name}`.`visitas_mensuales` (
    periodo STRING,
    email STRING,
    fecha_primera_visita_mes DATE,
    fecha_ultima_visita_mes DATE,
    visitas_mes BIGINT,
    id_proceso STRING,
    fecha_ejecucion TIMESTAMP
)
USING DELTA
PARTITIONED BY (periodo)
LOCATION '{visitas_mensuales_path}'
""")

print(f"Tabla creada o validada: {catalog_name}.{schema_name}.visitas_mensuales")
print(f"LOCATION: {visitas_mensuales_path}")


In [0]:
# COMMAND ----------

visitante_mensual_path = f"{gold_table_base_path}/visitante_mensual"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{schema_name}`.`visitante_mensual` (
    periodo STRING,
    email STRING,
    fecha_primera_visita DATE,
    fecha_ultima_visita DATE,
    visitas_totales BIGINT,
    visitas_anio_actual BIGINT,
    visitas_mes_actual BIGINT,
    periodo_baja STRING,
    id_proceso STRING,
    fecha_ejecucion TIMESTAMP
)
USING DELTA
PARTITIONED BY (periodo)
LOCATION '{visitante_mensual_path}'
""")

print(f"Tabla creada o validada: {catalog_name}.{schema_name}.visitante_mensual")
print(f"LOCATION: {visitante_mensual_path}")


In [0]:
# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE VIEW `{catalog_name}`.`{schema_name}`.`estadistica` AS
SELECT
    email,
    jyv,
    badmail,
    baja,
    fecha_envio,
    fecha_open,
    opens,
    opens_virales,
    fecha_click,
    clicks,
    clicks_virales,
    links,
    ips,
    navegadores,
    plataformas
FROM `{catalog_name}`.`{schema_name}`.`silver_reportes`
""")

print(f"Vista creada o reemplazada: {catalog_name}.{schema_name}.estadistica")


In [0]:
# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE VIEW `{catalog_name}`.`{schema_name}`.`visitante` AS
SELECT *
FROM `{catalog_name}`.`{schema_name}`.`visitante_mensual`
""")

print(f"Vista creada o reemplazada: {catalog_name}.{schema_name}.visitante")


In [0]:
# COMMAND ----------

print(f"Objetos lógicos creados o validados en {catalog_name}.{schema_name}")
print("Tablas Delta external/unmanaged creadas con LOCATION explícito.")
print("Volumes usados por este proceso: raw y backup.")
print("Auto Loader usará metadata en bronze/metadata/smart_data_ingestion.")
